In [1]:
# imports
import os
import time
import pickle
import pandas as pd
import argparse
import numpy as np
import xgboost as xgb
from sklearn.feature_selection import RFE, RFECV
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split

from utils_pyradiomics import preprocessing_train, preprocessing_test, get_optimal_threshold
#from preprocessing.radiomics_extraction import initialize_feature_extractor,generate_features_table
#from radiomics_pipeline.utils import preprocessing_train, preprocessing_test

In [2]:
# merge outcome with pyradiomics feature table

clinical_df = pd.read_csv(os.path.expanduser('~/project/xAI-in-NSCLC/NSCLC-Radiomics-Lung1.clinical-version3-Oct-2019.csv'))
features_df = pd.read_csv(os.path.expanduser('~/project/xAI-in-NSCLC/radiomics_features_per_slice.csv'))
print(clinical_df.shape, features_df.shape)

#fixed if you run the updated pyradiomics extraction.py
features_df.rename(columns={'patient_id': 'PatientID'}, inplace=True)
merged_df = pd.merge(features_df, clinical_df[['PatientID', 'Histology']], on='PatientID', how='left')
merged_df = merged_df.sort_values(by=['PatientID'], ascending=True)

(422, 10) (591, 662)


In [3]:
#fixed if you run the updated pyradiomics extraction.py
features_df.rename(columns={'patient_id': 'PatientID'}, inplace=True)
merged_df = features_df.merge(clinical_df[['PatientID', 'Histology']], on='PatientID', how='left')
merged_df = merged_df.sort_values(by=['PatientID'], ascending=True)

In [4]:
merged_df.head()
merged_df['Histology'].unique()
merged_df_clean = merged_df.dropna(subset=['Histology'])

In [5]:
merged_df_clean['Histology'].unique()

array(['large cell', 'squamous cell carcinoma', 'adenocarcinoma', 'nos'],
      dtype=object)

In [6]:
mapping = {'adenocarcinoma': 0, 'squamous cell carcinoma': 1, 'large cell': 2, 'nos':3 }
merged_df_clean['Histology'] = merged_df_clean['Histology'].map(mapping)
merged_df_clean['Histology'].unique()

/tmp/ipykernel_74167/2304721031.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged_df_clean['Histology'] = merged_df_clean['Histology'].map(mapping)


array([2, 1, 0, 3])

In [7]:
#test train split NOT by patient ID, but by slice

X = merged_df_clean.drop(columns=['PatientID', 'Histology'])
y = merged_df_clean['Histology']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

In [ ]:
# test train split by patient ID
temp_df = clinical_df.dropna(subset=['Histology'])
X = temp_df['PatientID']
y = temp_df['Histology']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=204, stratify=y)

#save patient IDs for deep learning test train split later
train_labels = X_train.unique()
test_labels = X_test.unique()

In [ ]:
#create test train split dfs for training
X_train = merged_df_clean.loc[merged_df_clean['PatientID'].isin(train_labels)]
y_train = X_train['Histology']
X_test = merged_df_clean.loc[merged_df_clean['PatientID'].isin(test_labels)]
y_test = X_test['Histology']

X_train = X_train.drop(columns=['PatientID', 'Histology', 'slice_no'])
X_test = X_test.drop(columns=['PatientID', 'Histology', 'slice_no'])

In [8]:
# preprocess training dataset first 
mean_std, selector, to_drop, decor_dataset_train = preprocessing_train(X_train)
decor_dataset_test = preprocessing_test(X_test, mean_std, selector, to_drop)
print('features processed. New shape of training dataset:', decor_dataset_train.shape, 'before:', X_train.shape)

features processed. New shape of training dataset: (365, 56) before: (365, 661)


In [9]:
print(f'y_train: {y_train.value_counts()/len(y_train)} N = {len(y_train)}')
print(f'y_test: {y_test.value_counts()/len(y_test)} N = {len(y_test)}')

y_train: Histology
1    0.531507
2    0.194521
0    0.191781
3    0.082192
Name: count, dtype: float64 N = 365
y_test: Histology
1    0.535032
2    0.191083
0    0.191083
3    0.082803
Name: count, dtype: float64 N = 157


In [10]:
# model definition
model = xgb.XGBClassifier(use_label_encoder=False, enable_categorical=True, colsample_bytree=1, eta=0.01, max_depth=4,
                              objective='multi:softprob', eval_metric='logloss', nthread=8, scale_pos_weight=1,
                              gamma=0.5, seed=204)

#note: can add 'device=cuda' to use GPU

#could you increase the number of threads for efficiency? To what number shoudl I increase it to?

#I think 32 is the limit for this environment.

In [11]:
# test to assume training time

start = time.time()
model.fit(X_train, y_train)
T_single = time.time() - start
print(T_single)


/home/coder/project/xAI-in-NSCLC/.venv/lib/python3.10/site-packages/xgboost/training.py:200: UserWarning: [11:57:31] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "scale_pos_weight", "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


9.142549276351929


In [12]:
# recursive feature elimination with cross validation:
min_features_to_select = 1  # Minimum number of features to consider
rfecv = RFECV(estimator=model, step=1, cv=StratifiedKFold(10),
              scoring='roc_auc_ovr_weighted',
              min_features_to_select=min_features_to_select)
rfecv.fit(decor_dataset_train, y_train)
support = rfecv.support_

/home/coder/project/xAI-in-NSCLC/.venv/lib/python3.10/site-packages/xgboost/training.py:200: UserWarning: [11:58:30] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "scale_pos_weight", "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/coder/project/xAI-in-NSCLC/.venv/lib/python3.10/site-packages/xgboost/training.py:200: UserWarning: [11:58:30] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "scale_pos_weight", "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/coder/project/xAI-in-NSCLC/.venv/lib/python3.10/site-packages/xgboost/training.py:200: UserWarning: [11:58:31] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "scale_pos_weight", "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/coder/project/xAI-in-NSCLC/.venv/lib/python3.10/site-packages/xgboost/training.py:200: UserWarning: [11:58:32] WARNING: /__w/xgboost/xgboost/src/

In [13]:
filtered_col = np.extract(support, np.array(decor_dataset_train.columns))
reduced_features_train_set = decor_dataset_train[filtered_col]

In [ ]:
from tqdm import tqdm


In [ ]:
#original cross validation search
param_test_xgb = {
        'max_depth': range(2, 4, 1),
        'min_child_weight': range(1, 6, 2),
        'gamma': [i * 0.1 for i in range(1, 10)], #was originally [i * 0.1 for i in range(1, 10)]
        'n_estimators': [int(x) for x in np.linspace(start=10, stop=1000, num=100)],
        'learning_rate': [10 ** (-i) for i in range(2, 3)] # was originally [10 ** (-i) for i in range(2, 7)]
    }

In [14]:
param_test_xgb = {
        'max_depth': range(3, 6, 1),
        'min_child_weight': [1],
        'gamma': [0, 0.2, 0.4], #was originally [i * 0.1 for i in range(1, 10)]
        'n_estimators': [200, 300, 400], #int(x) for x in np.linspace(start=200, stop=400, num=100)
    }

In [15]:
kfold = StratifiedKFold(n_splits=5, random_state=204, shuffle=True)
gsearch = GridSearchCV(model, param_grid=param_test_xgb, scoring='roc_auc_ovr_weighted', n_jobs=4, cv=kfold, verbose=1)

In [16]:
gsearch.fit(reduced_features_train_set, y_train)

Fitting 5 folds for each of 27 candidates, totalling 135 fits


/home/coder/project/xAI-in-NSCLC/.venv/lib/python3.10/site-packages/xgboost/training.py:200: UserWarning: [12:09:01] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "scale_pos_weight", "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/coder/project/xAI-in-NSCLC/.venv/lib/python3.10/site-packages/xgboost/training.py:200: UserWarning: [12:09:01] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "scale_pos_weight", "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/coder/project/xAI-in-NSCLC/.venv/lib/python3.10/site-packages/xgboost/training.py:200: UserWarning: [12:09:01] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "scale_pos_weight", "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/coder/project/xAI-in-NSCLC/.venv/lib/python3.10/site-packages/xgboost/training.py:200: UserWarning: [12:09:01] WARNING: /__w/xgboost/xgboost/src/

,estimator,"XGBClassifier...obs=None, ...)"
,param_grid,"{'gamma': [0, 0.2, ...], 'max_depth': range(3, 6), 'min_child_weight': [1], 'n_estimators': [200, 300, ...]}"
,scoring,'roc_auc_ovr_weighted'
,n_jobs,4
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,objective,'multi:softprob'


In [17]:
reduced_features_test_set = decor_dataset_test[filtered_col]

best_estimator = gsearch.best_estimator_

proba_train = best_estimator.predict_proba(reduced_features_train_set)
proba_test = best_estimator.predict_proba(reduced_features_test_set)


In [18]:
proba_train = rfecv.estimator_.predict_proba(reduced_features_train_set)
proba_test = rfecv.estimator_.predict_proba(reduced_features_test_set)

In [19]:
import sklearn

In [20]:
def get_multiclass_results(y_true, proba, label, average='weighted'):
    ## multiclass classification: choose argmax class and evaluate multiclass metrics
    y_true = np.asarray(y_true)
    y_pred = np.argmax(proba, axis=1)
    classes = np.unique(y_true)
    y_true_bin = sklearn.preprocessing.label_binarize(y_true, classes=classes)
    dict_results = {}
    dict_results["auc_ovr_weighted"] = sklearn.metrics.roc_auc_score(
        y_true_bin,
        np.asarray(proba),
        multi_class='ovr',
        average=average
    )
    dict_results["accuracy"] = sklearn.metrics.accuracy_score(y_true, y_pred)
    dict_results["precision"] = sklearn.metrics.precision_score(y_true, y_pred, average=average, zero_division=0)
    dict_results["recall"] = sklearn.metrics.recall_score(y_true, y_pred, average=average, zero_division=0)
    dict_results["f1 score"] = sklearn.metrics.f1_score(y_true, y_pred, average=average, zero_division=0)
    df_results = pd.DataFrame.from_dict([dict_results])
    df_results.index = [label]
    return df_results

In [21]:
results_train = get_multiclass_results(y_train, proba_train, "train")
results_test = get_multiclass_results(y_test, proba_test, "test")

In [22]:
results_train

,auc_ovr_weighted,accuracy,precision,recall,f1 score
train,0.99846,0.873973,0.895552,0.873973,0.868319


In [23]:
results_test

,auc_ovr_weighted,accuracy,precision,recall,f1 score
test,0.911961,0.745223,0.786947,0.745223,0.725726
